# 04 — Data Quality Report

1. **Expectations**: 3 reglas sobre trusted (2 univariadas + 1 cross-column) → `refined.data_quality_report`
2. **Reporte de ejecucion**: metricas globales del pipeline → `refined.pipeline_execution_report`

In [ ]:
%run ../config/pipeline_config

In [ ]:
import logging
import time
import json
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, TimestampType

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s - %(message)s")
logger = logging.getLogger("data_quality_report")

start_time = time.time()

In [ ]:
# Leer tabla trusted clean
df = spark.table(TRUSTED_CLEAN_TABLE)
total_records = df.count()
logger.info(f"Registros en trusted clean: {total_records:,}")

---
## Expectations

- 2 univariadas (rango de fare y distance)
- 1 cross-column (consistencia fare vs distance)

Umbral aceptable: >= 95%

In [ ]:
def evaluate_expectation(df, rule_name, rule_description, condition, threshold=0.95):
    total = df.count()
    passing = df.filter(condition).count()
    failing = total - passing
    pass_rate = round(passing / total, 4) if total > 0 else 0
    status = "PASS" if pass_rate >= threshold else "WARN"

    if status == "WARN":
        logger.warning(f"'{rule_name}': pass_rate={pass_rate:.2%} (bajo umbral {threshold:.0%})")
    else:
        logger.info(f"'{rule_name}': pass_rate={pass_rate:.2%} - {status}")

    return {
        "rule_name": rule_name,
        "rule_description": rule_description,
        "total_records": total,
        "passing_records": passing,
        "failing_records": failing,
        "pass_rate": pass_rate,
        "status": status,
        "evaluated_at": datetime.now()
    }

In [ ]:
exp1 = evaluate_expectation(
    df,
    rule_name="fare_amount_in_range",
    rule_description="fare_amount entre $2.50 (bajada de bandera) y $500",
    condition=(F.col("fare_amount") >= 2.5) & (F.col("fare_amount") <= 500)
)

exp2 = evaluate_expectation(
    df,
    rule_name="trip_distance_in_range",
    rule_description="trip_distance entre 0.1 y 100 millas",
    condition=(F.col("trip_distance") >= 0.1) & (F.col("trip_distance") <= 100)
)

# Cross-column: detecta combinaciones anomalas (ej: 0.2 mi con tarifa de $200)
df_for_cross = df.filter(F.col("trip_distance") > 0.1)
exp3 = evaluate_expectation(
    df_for_cross,
    rule_name="fare_distance_consistency",
    rule_description="fare/distance entre $1 y $100 por milla (viajes > 0.1 mi)",
    condition=(F.col("fare_amount") / F.col("trip_distance") >= 1) &
              (F.col("fare_amount") / F.col("trip_distance") <= 100)
)

expectations = [exp1, exp2, exp3]

## Data Quality Report

In [ ]:
schema = StructType([
    StructField("rule_name", StringType(), False),
    StructField("rule_description", StringType(), False),
    StructField("total_records", LongType(), False),
    StructField("passing_records", LongType(), False),
    StructField("failing_records", LongType(), False),
    StructField("pass_rate", DoubleType(), False),
    StructField("status", StringType(), False),
    StructField("evaluated_at", TimestampType(), False),
])

df_report = spark.createDataFrame([Row(**exp) for exp in expectations], schema=schema)

try:
    df_report.write.format("delta").mode("overwrite").saveAsTable(REFINED_DQ_REPORT)
    spark.sql(f"COMMENT ON TABLE {REFINED_DQ_REPORT} IS '3 expectations (2 univariadas + 1 cross-column) sobre trusted.'")

    props = ", ".join([f"'{k}' = '{v}'" for k, v in TABLE_PROPERTIES["refined"].items()])
    spark.sql(f"ALTER TABLE {REFINED_DQ_REPORT} SET TBLPROPERTIES ({props})")

    logger.info(f"'{REFINED_DQ_REPORT}' escrita.")
except Exception as e:
    logger.error(f"Error escribiendo DQ report: {e}")
    raise

df_report.show(truncate=False)

---
## Reporte de Ejecucion

In [ ]:
# Recopilar metricas de todas las capas
raw_taxi_count = spark.table(RAW_TAXI_TABLE).count()
raw_zones_count = spark.table(RAW_ZONES_TABLE).count()
trusted_clean_count = spark.table(TRUSTED_CLEAN_TABLE).count()
trusted_rejected_count = spark.table(TRUSTED_REJECTED_TABLE).count()
kpi1_count = spark.table(REFINED_KPI_DEMAND).count()
kpi2_count = spark.table(REFINED_KPI_EFFICIENCY).count()
kpi3_count = spark.table(REFINED_KPI_QUALITY_IMPACT).count()

# Breakdown de rechazos
rejection_breakdown = {
    row["rejection_reason"]: row["count"]
    for row in spark.table(TRUSTED_REJECTED_TABLE)
        .groupBy("rejection_reason").count()
        .collect()
}

# Resultados de expectations
expectations_summary = [
    {"rule": e["rule_name"], "pass_rate": e["pass_rate"], "status": e["status"]}
    for e in expectations
]

In [ ]:
# Construir reporte JSON completo
execution_report = {
    "pipeline_run_id": str(uuid.uuid4()),
    "execution_date": datetime.now().isoformat(),
    "status": "SUCCESS",
    "catalog": CATALOG_NAME,
    "dataset_month": "2023-01",
    "stages": [
        {
            "name": "01_raw_ingestion",
            "records_out": raw_taxi_count,
            "tables_created": [RAW_TAXI_TABLE, RAW_ZONES_TABLE],
            "details": f"{raw_taxi_count:,} taxi trips + {raw_zones_count} zones ingestados"
        },
        {
            "name": "02_trusted_transformation",
            "records_in": raw_taxi_count,
            "records_out": trusted_clean_count,
            "records_rejected": trusted_rejected_count,
            "discard_rate_pct": round(trusted_rejected_count / raw_taxi_count * 100, 2),
            "rejection_breakdown": rejection_breakdown
        },
        {
            "name": "03_refined_kpis",
            "kpi_demand_rows": kpi1_count,
            "kpi_efficiency_zones": kpi2_count,
            "kpi_quality_impact_rules": kpi3_count
        },
        {
            "name": "04_data_quality_report",
            "expectations": expectations_summary
        }
    ],
    "summary": {
        "total_records_processed": raw_taxi_count,
        "total_records_valid": trusted_clean_count,
        "total_records_discarded": trusted_rejected_count,
        "overall_discard_rate_pct": round(trusted_rejected_count / raw_taxi_count * 100, 2)
    }
}

report_json = json.dumps(execution_report, indent=2, default=str)
print("\n" + "="*70)
print("REPORTE DE EJECUCION DEL PIPELINE")
print("="*70)
print(report_json)
print("="*70)

In [ ]:
# Persistir reporte como tabla Delta para trazabilidad historica
report_row = Row(
    run_id=execution_report["pipeline_run_id"],
    execution_date=datetime.now(),
    status=execution_report["status"],
    total_processed=raw_taxi_count,
    total_valid=trusted_clean_count,
    total_discarded=trusted_rejected_count,
    discard_rate_pct=execution_report["summary"]["overall_discard_rate_pct"],
    report_json=report_json
)

df_exec = spark.createDataFrame([report_row])

try:
    df_exec.write.format("delta").mode("append").saveAsTable(REFINED_EXEC_REPORT)
    spark.sql(f"COMMENT ON TABLE {REFINED_EXEC_REPORT} IS 'Historico de ejecuciones del pipeline. Cada fila es una ejecucion con metricas y reporte JSON completo.'")
    logger.info(f"Reporte de ejecucion persistido en '{REFINED_EXEC_REPORT}'.")
except Exception as e:
    logger.error(f"Error persistiendo reporte de ejecucion: {e}")
    raise

elapsed = round(time.time() - start_time, 2)
logger.info(f"Data Quality Report completado en {elapsed}s.")

try:
    dbutils.notebook.exit(report_json)
except NameError:
    pass